# Aula 11 · pandas

Esta aula apresenta o [capítulo 11 do site](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/). A ideia central: **cada linha do pandas é um
laço que você já sabe escrever**. O pandas lê a tabela inteira e aplica filtro,
transformação, acumulador, contagem e agrupamento à coluna toda de uma vez — e quem
conhece o laço é quem consegue conferir o resultado.

**Ao fim da aula você consegue:**

1. ler um CSV num DataFrame e pegar uma coluna;
2. filtrar com máscara, criar coluna e resumir (`mean`, `max`, `value_counts`);
3. agrupar com `groupby`, ordenar e calcular a disponibilidade de cada enlace.

**Roteiro:** 🔥 aquecimento · 📟 chamado · 1. `read_csv` e o DataFrame · 2. o
reencontro · 3. filtrar · 4. nova coluna · 5. resumir · 6. agrupar · 7. ordenar ·
8. datas · 9. do log para a tabela · 📟 resolvendo o chamado · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

A célula ⚙️ desta aula também **baixa o mês de medições** (`medicoes_mes.csv`,
3 720 linhas: cinco OLTs, de hora em hora, em março de 2026) do site do curso.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
import os
import urllib.request

ARQUIVO = "medicoes_mes.csv"
if not os.path.exists(ARQUIVO):
    urllib.request.urlretrieve(
        "https://lacouth.github.io/python_telecom-site/dados/medicoes_mes.csv", ARQUIVO)
print(ARQUIVO, "pronto")

## 🔥 Aquecimento — da aula passada

Sem rodar nada: o que este trecho imprime?

```python
from dataclasses import dataclass

@dataclass
class Enlace:
    nome: str
    disponibilidade: float

    def cumpre(self, sla):
        return self.disponibilidade >= sla

enlaces = [Enlace("A", 99.9), Enlace("B", 98.2), Enlace("C", 99.5)]
ruins = []
for e in enlaces:
    if not e.cumpre(99.5):
        ruins.append(e.nome)
print(ruins)
```

<details>
<summary><b>Resposta</b></summary>

Imprime `['B']`. O `C` tem exatamente 99,5 e a comparação é `>=`, então cumpre. É o
filtro de sempre, com um método da dataclass no `if` — e hoje o mesmo filtro vai
ser escrito em uma linha.

</details>

> 💡 **Pense assim: da panela para a batedeira.**
>
> Uma **biblioteca** é um conjunto de ferramentas prontas, escritas e testadas por
> outras pessoas, que se instala e se importa como o `math`. Quem sabe bater massa na
> mão entende o que a batedeira faz — e percebe na hora quando ela está batendo
> errado. Você aprendeu a bater na mão (os laços); o pandas é a batedeira.

## 📟 O chamado de hoje

> **Chamado #1103 — NOC Maré Net**
>
> *"Estagiário, fechamento do mês: o contrato promete **99,5% de disponibilidade**
> em cada OLT. Preciso saber quais descumpriram e com quanto. Até hoje alguém abre o
> CSV de 3 720 linhas na planilha e monta uma tabela dinâmica — e todo mês sai um
> número diferente dependendo de quem faz."*

No fim da aula você calcula isso em poucas linhas — sempre do mesmo jeito.

## 1. `read_csv` e o DataFrame

`pd.read_csv` lê o CSV inteiro numa **tabela**, o **DataFrame**. Uma coluna sozinha
é uma **Series**: uma lista de valores com nome. E a conversão de texto para número,
que você fazia com `float()`, já vem feita.

📖 [capítulo 11 · `read_csv` e o DataFrame](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#read-csv-e-o-dataframe)

> 💡 **Pense assim: a planilha que se programa.**
>
> Um DataFrame é uma planilha dentro do Python: linhas numeradas, colunas com nome
> no cabeçalho, uma célula em cada cruzamento. A diferença é que, em vez de clicar e
> arrastar fórmulas, você escreve a operação uma vez — e ela vale para a coluna
> inteira, com três mil ou três milhões de linhas.

**✍️ Passo 1.** Escreva `import pandas as pd` e `df = pd.read_csv("medicoes_mes.csv")`. Imprima
`df.shape` e `df.head()`.

In [ ]:
# ✍️ passo 1

**Preveja:** quantas linhas tem a tabela? (Dica: 5 OLTs, 24 horas por dia, 31 dias.)

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`(3720, 5)` — 5 × 24 × 31 linhas e 5 colunas —, e as cinco primeiras linhas com um
número à esquerda, o **índice**. O `head()` é o jeito de "espiar" uma tabela grande
sem imprimir tudo.

</details>

**✍️ Passo 2.** Pegue uma coluna: `potencia = df["potencia_dbm"]` e imprima `potencia.head(3)`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que aparece além dos três valores?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Os três valores com o índice à esquerda, o nome da coluna e o tipo (`float64`): a
Series sabe que é número. Nenhum `float()` foi escrito.

</details>

## 2. O reencontro

A média da potência, com o laço do capítulo 7 e com o pandas: o mesmo número.

📖 [capítulo 11 · O reencontro](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#o-reencontro)

> 💡 **Pense assim: a célula em branco da lista de presença.**
>
> Na lista de presença, a célula em branco não quer dizer "zero faltas" nem "zero
> presenças": quer dizer **não anotado**. O `NaN` é isso. Por isso o pandas não o trata
> como zero numa média — contá-lo como zero puxaria a média para baixo com uma
> medição que nunca existiu.

**✍️ Passo 3.** Imprima `round(df["potencia_dbm"].mean(), 2)`. Depois imprima `len(df)` e
`df["potencia_dbm"].count()`.

In [ ]:
# ✍️ passo 3

**Preveja:** por que `len` e `count` dão números diferentes?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A média sai `-21.91`; `len` dá `3720` e `count` dá `3704`. As 16 horas em que um
equipamento estava fora do ar não têm potência: a célula vazia virou **`NaN`** ("não
é um número"), e as contas do pandas a deixam de fora sozinhas — exatamente o
`if registro["potencia_dbm"] == "": continue` que o laço precisava.

</details>

## 3. Filtrar

Comparar uma coluna inteira dá uma coluna de `True`/`False` — a **máscara**. A
máscara entre colchetes escolhe as linhas: é o padrão **filtro**.

📖 [capítulo 11 · Filtrar](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#filtrar)

> 💡 **Pense assim: o marca-texto.**
>
> A máscara é passar o marca-texto nas linhas da planilha que atendem à condição:
> cada linha fica marcada (`True`) ou não (`False`). `df[mascara]` é recortar e
> ficar só com as linhas marcadas. Contar as marcadas (somar a máscara) é contar
> quantas linhas atendem à condição.

In [ ]:
# 📦 dados prontos — só rode esta célula
# a mesma tabela do passo 1 — garante que ela está carregada para os exercícios
import pandas as pd

df = pd.read_csv("medicoes_mes.csv")

**✍️ Passo 4.** Faça `fora = df["estado"] == "DOWN"` e imprima `fora.head(3)` e `fora.sum()`.

In [ ]:
# ✍️ passo 4

**Preveja:** o que é somar uma coluna de `True`/`False`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`fora.head(3)` mostra `False`, `False`, `False`; `fora.sum()` dá `16`. Cada `True`
conta como 1 — somar a máscara é **contar** as linhas que atendem.

</details>

**✍️ Passo 5.** Imprima `df[fora].head()`. Depois, duas condições:
`df[(df["enlace"] == "OLT-SUL-03") & (df["potencia_dbm"] < -25)]` e imprima o `len`
disso.

In [ ]:
# ✍️ passo 5

**Preveja:** por que cada condição tem os próprios parênteses?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A primeira mostra as primeiras horas fora do ar (com `NaN` na potência); a segunda dá
`244` horas da OLT-SUL-03 abaixo de −25 dBm. O `&` quer dizer "e" entre colunas, e
**cada condição precisa de parênteses** — sem eles, o Python junta as peças na ordem
errada.

</details>

> ⚠️ **Armadilha.** Usar `and` no lugar de `&`. O `and` compara **um** valor com outro; com colunas
> inteiras, dá `ValueError: The truth value of a Series is ambiguous`. Com tabela:
> `&` (e), `|` (ou), e parênteses em cada condição.

### 🎯 Sua vez — Horas em alerta

Escreva `horas_em_alerta(df, enlace, limite)`, que devolve **quantas** medições
daquele enlace ficaram abaixo do `limite`, como `int`.

In [ ]:
def horas_em_alerta(df, enlace, limite):
    # sua solução aqui
    pass

In [ ]:
confere(horas_em_alerta, [
    ((df, "OLT-SUL-03", -25), 244),
    ((df, "OLT-CENTRO-01", -25), 0),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas condições com `&`, cada uma entre parênteses, e a soma da máscara convertida
com `int(...)`.

</details>

## 4. Nova coluna

Uma conta escrita com a coluna inteira vale para todas as linhas; atribuir a um nome
novo cria a coluna. É o padrão **transformação**.

📖 [capítulo 11 · Nova coluna](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#nova-coluna)

**✍️ Passo 6.** Crie `df["margem_db"] = (df["potencia_dbm"] + 27).round(2)` e
`df["no_ar"] = df["estado"] == "UP"`. Imprima
`df[["enlace", "potencia_dbm", "margem_db", "no_ar"]].head(3)`.

In [ ]:
# ✍️ passo 6

**Preveja:** para que servem os **dois** pares de colchetes na última linha?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A tabela mostra só as quatro colunas pedidas: com uma lista de nomes dentro dos
colchetes, escolhe-se um pedaço da tabela. A margem é a potência menos a
sensibilidade (−27 dBm) em cada linha, e `no_ar` é `True` em toda hora com estado
`UP` — essa coluna vai virar a disponibilidade daqui a pouco.

</details>

## 5. Resumir

`mean`, `min`, `max` e `sum` são o acumulador e o extremo; `value_counts` é a
contagem com dicionário do capítulo 5.

📖 [capítulo 11 · Resumir](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#resumir)

**✍️ Passo 7.** Imprima `df["trafego_mbps"].max()` e `df["estado"].value_counts()`.

In [ ]:
# ✍️ passo 7

**Preveja:** o que o `value_counts` mostra?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`981.4` (o maior tráfego do mês) e a contagem de cada valor da coluna: `UP 3704` e
`DOWN 16`. É o `contagem[chave] = contagem.get(chave, 0) + 1` da Aula 05, numa
palavra.

</details>

### 🎯 Sua vez — O pico de um enlace

Escreva `pico_do_enlace(df, enlace)`, que devolve o **maior tráfego** registrado por
aquele enlace, como `float`.

In [ ]:
def pico_do_enlace(df, enlace):
    # sua solução aqui
    pass

In [ ]:
confere(pico_do_enlace, [
    ((df, "OLT-CENTRO-01"), 981.4),
    ((df, "OLT-LESTE-04"), 452.5),
])

<details>
<summary><b>💡 Dica</b></summary>

Filtre as linhas do enlace, pegue a coluna `trafego_mbps` e o `.max()` dela — com
`float(...)` em volta.

</details>

## 6. Agrupar

`groupby("enlace")` separa a tabela em um pedaço por enlace; o que vem depois diz o
que fazer com cada pedaço. É o agrupamento do capítulo 5 — agora, numa linha.

📖 [capítulo 11 · Agrupar](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#agrupar)

> 💡 **Pense assim: as pilhas de provas.**
>
> Com as provas de todas as turmas misturadas, a coordenadora separa uma pilha por
> turma (`groupby("turma")`) e, em cada pilha, tira a média das notas (`.mean()`) ou
> conta quantas provas há (`.size()`). O resultado é uma linha por turma.

> 📡 **Na rede: KPI e disponibilidade.**
>
> Um **KPI** é um número que a operação acompanha sempre, como o painel do carro. A
> **disponibilidade** é o mais importante deles: a porcentagem do tempo em que o
> serviço funcionou. Março tem 744 horas; ficar 8 horas fora dá 98,925%. Parece
> muito, mas um contrato (**SLA**) de 99,5% permite só 3,7 horas fora nesse mês — e
> quem descumpre devolve dinheiro ao cliente. Veja
> [Disponibilidade, SLA e KPI](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/#disponibilidade-sla-e-kpi).

**✍️ Passo 8.** Imprima `df[df["estado"] == "DOWN"].groupby("enlace").size()`.

In [ ]:
# ✍️ passo 8

**Preveja:** o que esta linha responde, em português?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Quantas horas cada enlace ficou fora do ar**: OLT-LESTE-04 com 8, OLT-NORTE-02 com
2, OLT-OESTE-05 com 6. Filtra (só `DOWN`), agrupa (por enlace), conta (`size`). Os
enlaces que nunca caíram não aparecem, porque não sobrou nenhuma linha deles depois
do filtro.

</details>

**✍️ Passo 9.** Imprima `(df.groupby("enlace")["no_ar"].mean() * 100).round(3)`.

In [ ]:
# ✍️ passo 9

**Preveja:** o que é a **média** de uma coluna de `True`/`False`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

É a **fração** de `True` — cada `True` conta 1 e cada `False` conta 0. Vezes 100, é a
porcentagem do tempo no ar: a **disponibilidade**. A OLT-LESTE-04 ficou em 98,925%:
8 horas fora num mês de 744.

</details>

### 🎯 Sua vez — Tráfego médio por enlace

Escreva `media_trafego(df)`, que devolve um **dicionário** enlace → tráfego médio,
com 1 casa.

In [ ]:
def media_trafego(df):
    # sua solução aqui
    pass

In [ ]:
confere(media_trafego, [
    ((df,), {"OLT-CENTRO-01": 431.2, "OLT-LESTE-04": 199.4, "OLT-NORTE-02": 283.9,
             "OLT-OESTE-05": 234.3, "OLT-SUL-03": 320.2}),
])

<details>
<summary><b>💡 Dica</b></summary>

`groupby("enlace")["trafego_mbps"].mean()`, depois `.round(1)` e `.to_dict()` — o
último transforma o resultado num dicionário do Python.

</details>

## 7. Ordenar e top-N

`sort_values(coluna, ascending=False)` ordena do maior para o menor, e `head(n)` pega
os `n` primeiros — o top-N da Aula 05, sem função de `key=`.

📖 [capítulo 11 · Ordenar e top-N](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#ordenar-e-top-n)

**✍️ Passo 10.** Imprima `df.sort_values("trafego_mbps", ascending=False).head(3)`.

In [ ]:
# ✍️ passo 10

**Preveja:** as três horas de maior tráfego são de enlaces diferentes?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

As três são da OLT-CENTRO-01, todas por volta das 21h — ela é, de longe, o enlace com
mais tráfego. O top-N mostra a linha inteira, com o momento e o enlace de cada pico.

</details>

## 8. Datas

`pd.to_datetime` é o `strptime` da Aula 08 aplicado à coluna inteira; depois dele,
`.dt.hour` e `.dt.day` tiram a hora e o dia de cada linha.

📖 [capítulo 11 · Datas](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#datas)

> 📡 **Na rede: tráfego e janela de manutenção.**
>
> O **tráfego** é a quantidade de dados que passa pelo enlace por segundo, em Mbps
> (megabits por segundo). Ele sobe à noite, quando todo mundo chega em casa e assiste
> a vídeos, e desce de madrugada. A **janela de manutenção** é o horário combinado
> para mexer na rede (trocar uma placa, atualizar o sistema): escolhe-se o vale, para
> afetar o menor número de clientes. Veja
> [Tráfego e utilização](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/#trafego-e-utilizacao-bits-por-segundo).

**✍️ Passo 11.** Faça `df["timestamp"] = pd.to_datetime(df["timestamp"])` e
`df["hora"] = df["timestamp"].dt.hour`. Imprima
`df.groupby("hora")["trafego_mbps"].mean().round(0)`.

In [ ]:
# ✍️ passo 11

**Preveja:** em que hora do dia a rede tem mais tráfego? E menos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

As 24 médias, uma por hora: o vale às 4h (69 Mbit/s) e o pico às 21h (574 Mbit/s).
É o **perfil diário** de uma rede residencial — e é ele que manda a janela de
manutenção para a madrugada.

</details>

### 🎯 Sua vez — As horas de pico

Escreva `horas_de_pico(df, n)`, que devolve a **lista** das `n` horas do dia com maior
tráfego médio, da maior para a menor. A coluna `hora` já existe (passo anterior); se
ela ainda não existir no seu `df`, rode o passo acima.

In [ ]:
def horas_de_pico(df, n):
    # sua solução aqui
    pass

In [ ]:
df["hora"] = pd.to_datetime(df["timestamp"]).dt.hour
confere(horas_de_pico, [((df, 3), [21, 20, 19]), ((df, 1), [21])])

<details>
<summary><b>💡 Dica</b></summary>

Agrupe pela hora, tire a média do tráfego, `sort_values(ascending=False)`, `head(n)`,
e a lista das horas é o **índice** do resultado: `.index.tolist()`.

</details>

## 9. Do log para a tabela

A lista de dicionários que o parser da Aula 09 devolve vira uma tabela numa linha:
cada dicionário é uma linha, cada chave uma coluna.

📖 [capítulo 11 · Do log para a tabela](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/#do-log-para-a-tabela)

In [ ]:
# 📦 dados prontos — só rode esta célula
alarmes = [
    {"severidade": "CRITICAL", "equipamento": "OLT-CENTRO-01"},
    {"severidade": "ERROR", "equipamento": "SWITCH-NORTE-02"},
    {"severidade": "WARNING", "equipamento": "RADIO-OESTE-01"},
    {"severidade": "CRITICAL", "equipamento": "SWITCH-CENTRO-01"},
    {"severidade": "CRITICAL", "equipamento": "RADIO-OESTE-01"},
]

**✍️ Passo 12.** Faça `tabela = pd.DataFrame(alarmes)` e imprima `tabela["severidade"].value_counts()`.

In [ ]:
# ✍️ passo 12

**Preveja:** quantas linhas e colunas tem a `tabela`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cinco linhas (uma por alarme) e duas colunas (uma por chave). O `value_counts` dá
`CRITICAL 3`, `ERROR 1`, `WARNING 1`: normalizar com o parser da Aula 09 e analisar
com pandas é a divisão de trabalho de uma ferramenta de análise de log de verdade.

</details>

## 📟 Resolvendo o chamado

### 🎯 Sua vez — Quem descumpriu o SLA

Escreva `fora_do_sla(df, sla)`, que devolve um **dicionário** com os enlaces cuja
disponibilidade ficou **abaixo** do `sla`, cada um com a disponibilidade em %
arredondada com **3 casas**.

In [ ]:
def fora_do_sla(df, sla):
    # sua solução aqui
    pass

In [ ]:
confere(fora_do_sla, [
    ((df, 99.5), {"OLT-LESTE-04": 98.925, "OLT-OESTE-05": 99.194}),
    ((df, 99.0), {"OLT-LESTE-04": 98.925}),
    ((df, 90.0), {}),
])

<details>
<summary><b>💡 Dica</b></summary>

Três passos: a coluna `no_ar` (`df["estado"] == "UP"`); a disponibilidade com
`groupby` + `mean() * 100` + `round(3)`; e o filtro `disp[disp < sla]` com
`.to_dict()` no fim.

</details>

**Resposta ao chamado:** OLT-LESTE-04 (98,925%, 8 horas fora) e OLT-OESTE-05
(99,194%, 6 horas fora) descumpriram o SLA de 99,5%. A OLT-NORTE-02 teve duas horas
fora e ficou em 99,731% — dentro. E o cálculo dá o mesmo resultado não importa quem
rode.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** `df[df["estado"] == "DOWN"]` corresponde a qual padrão dos laços?
a) acumulador  b) filtro  c) transformação  d) extremo

<details>
<summary><b>Resposta da 1</b></summary>

**b**. A máscara escolhe as linhas que atendem à condição.

</details>

**2.** Para juntar duas condições num filtro do pandas, escreve-se:
a) `cond1 and cond2`  b) `(cond1) & (cond2)`  c) `cond1 & cond2` sem parênteses
d) `cond1, cond2`

<details>
<summary><b>Resposta da 2</b></summary>

**b**. `and` dá erro com colunas, e sem parênteses a ordem das operações sai errada.

</details>

**3.** A média de uma coluna com `[True, True, False, True]` vale:
a) `3`  b) `0.75`  c) `True`  d) erro

<details>
<summary><b>Resposta da 3</b></summary>

**b**. `True` conta 1 e `False` conta 0: 3 de 4. É assim que se calcula a
disponibilidade.

</details>

## 🏠 Para casa

- [Lista 11](https://lacouth.github.io/python_telecom-site/listas/lista11/) —
  pandas, com testes automáticos no Colab.
- Releia o [capítulo 11 do site](https://lacouth.github.io/python_telecom-site/unidade6-dados/11-pandas/).
- **Na próxima aula:** mini-teste sobre esta aula (máscara, `groupby` e
  disponibilidade).